# 01 · Forecasting data audit (US Embassy station, OpenAQ 8160)

**Goal:** check that the hourly PM2.5 record is good enough for next-day forecasting, and lock the decisions the models depend on.

**Needs:** OpenAQ archive files for location 8160 anywhere under `data/raw/pm25/` (e.g. `data/raw/pm25/openaq/8160/`).
For a dry run without real data: `python scripts/make_synthetic_data.py` from the repo root (fake data, delete it afterwards).

All logic lives in `src/uaq/` (`io.py`, `daily.py`, `audit_forecast.py`, `thresholds.py`, `baselines.py`); this notebook runs it step by step.

In [ ]:
# Setup: works whether or not you ran `pip install -e .`
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from uaq.config import load_config, ROOT
cfg = load_config()
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
%matplotlib inline
print("Project root:", ROOT)

## 1 · Study design from the config
Every date here is locked. The test winter (2024/25) is never scored in this notebook.

In [ ]:
fp = cfg["forecast"]["periods"]
display(pd.DataFrame(fp["split"]).T.rename_axis("split"))
print("Core:", fp["core"], "| Pre-ban sensitivity:", fp["pre_ban_sensitivity"])
print("Smog season months:", cfg["smog_months"], "| Raw-coal ban:", cfg["structural_break"])

## 2 · Load the hourly data
Timestamps are converted to Ulaanbaatar local time. Negative values become missing; values above the plausibility bound are **flagged, not deleted**, because winter peaks are genuinely extreme.

In [ ]:
from uaq.io import load_hourly
hourly = load_hourly(cfg)
print(hourly.attrs)
print("Rows:", len(hourly), "| from", hourly["ts_local"].min(), "to", hourly["ts_local"].max())
hourly.head()

## 3 · Hourly → daily means
A day counts only if at least 18 of 24 hours were observed; otherwise it is **missing**, never treated as clean air.

In [ ]:
from uaq.daily import to_daily
daily = to_daily(hourly, cfg["quality"]["min_hours_per_day"])
print(f"Valid days: {daily['valid'].sum()} of {len(daily)} ({daily['valid'].mean():.1%})")
daily[["n_hours", "pm25_daily", "max_hourly"]].describe().round(1)

In [ ]:
from uaq.periods import assign_split
fig, ax = plt.subplots(figsize=(12, 3.5))
for sid, g in daily.groupby("station_id"):
    ax.plot(g["date"], g["pm25_daily"], lw=0.6, label=sid)
colors = {"train": "#4C72B0", "val": "#DD8452", "test": "#C44E52"}
for name, s in fp["split"].items():
    ax.axvspan(pd.Timestamp(s["start"]), pd.Timestamp(s["end"]), color=colors[name], alpha=0.08, label=name)
ax.axvline(pd.Timestamp(cfg["structural_break"]), color="k", ls="--", lw=1, label="coal ban")
ax.set_ylabel("Daily mean PM2.5 (µg/m³)"); ax.legend(ncol=5, fontsize=8); ax.set_title("Daily PM2.5 and the locked splits")
plt.show()

## 4 · Coverage by smog season
Share of Nov–Feb days with a valid daily mean. The station is usable if training holds at least 240 valid smog-season days in total (two full winters’ worth; partial winters count, because the model trains on every valid day) and the validation and test winters each have ≥75% coverage. good_train_winters is reported for the write-up only.

In [ ]:
from uaq import audit_forecast as af
cov = af.coverage_table(daily, cfg)
af.plot_coverage(cov, cfg["quality"]["min_valid_frac"]); plt.show()
elig = af.eligibility(cov, cfg)
elig

## 5 · Before and after the raw-coal ban
Why the core study starts in July 2019: the level shifts at the ban, so pre-ban winters are kept only for the optional sensitivity experiment (H5).

In [ ]:
af.plot_season_means(daily, cfg); plt.show()

## 6 · Daily cycle in winter
Motivates the hourly features (yesterday evening's mean and maximum). Hours are local time.

In [ ]:
from uaq.periods import in_months
h = hourly[in_months(hourly["ts_local"].dt.tz_localize(None), cfg["smog_months"]).values]
prof = h.groupby(h["ts_local"].dt.hour)["pm25"].quantile([0.25, 0.5, 0.75]).unstack()
ax = prof[0.5].plot(figsize=(7, 3), label="median")
ax.fill_between(prof.index, prof[0.25], prof[0.75], alpha=0.3, label="25–75%")
ax.set_xlabel("hour of day"); ax.set_ylabel("PM2.5 (µg/m³)"); ax.legend(); ax.set_title("Smog-season daily cycle")
plt.show()

## 7 · Event threshold (training smog-season days only)
Rule: the lowest official line exceeded on ≤60% of training days; otherwise the training 75th percentile, labelled study-defined.

In [ ]:
from uaq import thresholds
rates = af.threshold_table(daily, cfg)
display(rates.round(3))
thr_name, thr, why = thresholds.choose(rates, cfg)
print(f"Chosen: {thr_name} = {thr} µg/m³\n{why}")

## 8 · Baselines the neural networks must beat
Persistence (tomorrow = today), 3-day mean, and climatology (fitted on training only). Validation scores are the bar; the test winter is not scored here.

In [ ]:
stations = elig.index[elig["eligible"]].tolist()
scores = af.baseline_scores(daily, stations, thr, cfg)
scores.round(3)

## 9 · Save the audit report
Writes `outputs/audit_forecast/audit_report.md` plus the tables and figures above.
Record the decisions (usable station, threshold, baseline scores) in the plan doc.

In [ ]:
report = af.run()
print(report.read_text()[:1500])